# Janus 04 Valuation｜第21版：FinMind 免費資料真實補收（斷點續跑）

**這份不是盤點版：預設 `RUN_FETCH=True`，會呼叫 FinMind 官方 API。** 每輪預設最多 **25 次實際 HTTP 請求**；成功一筆就把回應原始 bytes、SHA256、UTC 時間、無密碼 receipt 寫到 Drive。同樣 Notebook 再按「全部執行」會驗 SHA，跳過已完成，接著抓後續未完成的請求。遇免費額度限制會停止，本輪未完成的留待下一輪。

**先決條件：**已在同一份 Google Drive 白名單 `janusChatGPT/模型研究/valuation/preflight_30_valid_monthends_20261024` 執行第20版 Notebook 並產生六個檔案：`00_summary.json`、`01...csv`、`02...csv`、`03...csv`、`04...csv`、`master_sources.json`；完整檔名在下一個儲存格與本版程式裡。不需要重新執行盤點。預先準備你的 **FinMind 免費帳號 Token**（程式會安全地詢問，不保存在 GDrive）。

**抓取範圍：**僅限第20版清冊中未達 PE/PB 30 個不同月末正值的官方現況普通股候選，對缺月/空值用免費 `TaiwanStockPER` 查核；整季無資料且與此研究相關者，定向查損益及資產負債表。**PE≤0、PB≤0 不當作可補正值；IPO 前月份不發出請求**。最多往回 60 個月，且只重抓與清冊交集的 stock×dataset 日期範圍，不是全市場抓取。

**治理：**研究原始來源候選 ≠ 合法 PIT/真正月底正式估值 ≠ 可訓練或可發布模型。`publication_allowed=false`；不修改任何 canonical/CURRENT/ACTIVE/Registry、現役服務、使用者持股。歷史的首次公告版本／授權條件需另查證。FinMind 官方資料集與 Bearer header 文件： [PER/PBR](https://finmind.github.io/tutor/TaiwanMarket/Technical/)／[快速開始與免費額度](https://finmind.github.io/quickstart/)。


In [ ]:
# 1. 掛載指定 Drive 並設定每輪額度；可直接「執行全部」
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, hashlib, json, getpass, requests, time
ROOT=Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
SOURCE_DIR=ROOT/'模型研究'/'valuation'/'preflight_30_valid_monthends_20261024'
DEST=ROOT/'模型研究'/'valuation'/'backfill_candidates'/'FINMIND_MIN30_REAL_20261024'
RUN_FETCH=True                         # 真實呼叫 FinMind；非 dry-run
MAX_CALLS_PER_RUN=25                  # 每輪最多25次，含重試；增加前先確認配額
SLEEP_SECONDS=8.0                    # 同一輪限速，每小時理論上<450次
MAX_LOOKBACK_MONTHS=60               # 不是只抓兩年；最多回看60月，先用已有36月
RETRY_ONCE=False                     # 預設不在同輪重試，留待下次重跑
assert ROOT.is_dir() and ROOT.name=='janusChatGPT'
assert SOURCE_DIR.is_dir(), '找不到第20版結果：'+str(SOURCE_DIR)
assert MAX_CALLS_PER_RUN>=1 and SLEEP_SECONDS>=7.2
assert DEST.resolve().is_relative_to(ROOT.resolve())
print('輸入結果：',SOURCE_DIR)
print('候選回應輸出：',DEST)
print('FinMind API 實抓：',RUN_FETCH,'本輪請求上限：',MAX_CALLS_PER_RUN)


In [ ]:
# 2. 第21版內嵌完整核心（不依賴下載 GitHub main 或本機 .py）
"""Janus Valuation 04 research-only FinMind scoped backfill. No canonical writes."""
from __future__ import annotations
import csv
import datetime as dt
import hashlib
import io
import json
import math
import os
import re
import time
from collections import defaultdict
from pathlib import Path

BASE_URL = 'https://api.finmindtrade.com/api/v4/data'
OFFICIAL = {'twse': 'https://openapi.twse.com.tw/v1/opendata/t187ap03_L',
            'tpex': 'https://mopsfin.twse.com.tw/opendata/t187ap03_O.csv'}
DATASETS = {'valuation': 'TaiwanStockPER',
            'income': 'TaiwanStockFinancialStatements',
            'balance': 'TaiwanStockBalanceSheet'}
REQUIRED = ('00_summary.json', '01_普通股估值與季報逐股核對.csv',
            '02_PE_PB_30有效月末缺口逐月查核.csv',
            '03_30有效月末_股票分支短缺清單.csv',
            '04_財報無紀錄_相關性待核查.csv', 'master_sources.json')
EXPECTED_RELEASE = 'release-0bbae8d9c1ca7d8d'
EXPECTED_CURRENT_SHA = '0b64ac6927805fb98260121da94624d7e08ac423f0aa13b8349d4c304f0135cd'
EXPECTED_MANIFEST_SHA = '7b80d441efabbc307d359ea5d15cd9202ccee5e2930cf4c3cc337496c95eb542'


def sha(data: bytes) -> str: return hashlib.sha256(data).hexdigest()
def utc(): return dt.datetime.now(dt.timezone.utc).isoformat()

def atomic_bytes(path: Path, data: bytes):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_name(path.name+'.part')
    with open(tmp, 'wb') as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def atomic_json(path: Path, data):
    atomic_bytes(path, (json.dumps(data, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False)+'\n').encode())

def records(path: Path):
    with open(path, 'r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

def atomic_csv(path: Path, rows: list[dict], fields: list[str]):
    f=io.StringIO(newline=''); writer=csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    writer.writeheader(); writer.writerows(rows)
    atomic_bytes(path, f.getvalue().encode('utf-8-sig'))

def month_ord(s):
    m=re.fullmatch(r'(\d{4})-(\d\d)', str(s))
    if not m: raise ValueError('bad month: '+str(s))
    y,n=map(int,m.groups());
    if n not in range(1,13): raise ValueError('bad month')
    return y*12+n-1

def ymonth(n): return f'{n//12:04d}-{n%12+1:02d}'
def first_day(m): return str(m)+'-01'
def last_day(m):
    o=month_ord(m)
    return str(dt.date.fromisoformat(first_day(ymonth(o+1)))-dt.timedelta(days=1))

def quarter_end(q):
    if not re.fullmatch(r'\d{4}Q[1-4]', str(q)): raise ValueError('bad quarter')
    y=int(q[:4]); n=int(q[-1])*3
    return last_day(f'{y}-{n:02d}')

def listing_date(value):
    s=re.sub(r'[^0-9]', '', str(value or ''))
    if len(s)==7: s=str(int(s[:3])+1911)+s[3:]
    if len(s)!=8: return None
    try:
        d=dt.date(int(s[:4]),int(s[4:6]),int(s[6:8]));
        if d<dt.date(1990,1,1) or d>dt.date(2026,10,24):return None
        return d.isoformat()
    except ValueError: return None

def parse_official_master(market: str, raw: bytes):
    if market=='twse':
        j=json.loads(raw)
        if not isinstance(j,list): raise ValueError('TWSE master not list')
        rows=j
    else:
        rows=None
        for enc in ('utf-8-sig','cp950','big5'):
            try:
                rows=list(csv.DictReader(io.StringIO(raw.decode(enc))));break
            except UnicodeDecodeError: pass
        if rows is None: raise ValueError('TPEx master encoding')
    if not rows or not any('公司代號' in r for r in rows): raise ValueError('official roster schema')
    result={}
    for r in rows:
        symbol=str(r.get('公司代號','')).strip(); name=str(r.get('公司名稱','')).strip()
        if not re.fullmatch(r'[1-9][0-9]{3}',symbol) or symbol.startswith('91'): continue
        if re.search('存託|TDR|特別股|受益證券|基金|權證|ETF|ETN',name,re.I): continue
        ld=None
        for k,v in r.items():
            if '上市日期' in str(k) or '上櫃日期' in str(k) or str(k).strip() in ('上市日','上櫃日'):
                ld=listing_date(v)
                if ld: break
        result[(market,symbol)]={'name':name,'listing_date':ld}
    if not result: raise ValueError('official roster produced zero candidates')
    return result

def read_sources(source_dir: Path):
    source_dir=Path(source_dir).resolve()
    for n in REQUIRED:
        if not (source_dir/n).is_file(): raise FileNotFoundError(f'需要第20版盤點產物：{source_dir/n}')
    summary=json.loads((source_dir/REQUIRED[0]).read_text('utf-8'))
    src=summary.get('source',{})
    if src.get('release')!=EXPECTED_RELEASE or src.get('current_sha256')!=EXPECTED_CURRENT_SHA or src.get('manifest_sha256')!=EXPECTED_MANIFEST_SHA:
        raise ValueError('研究資料版本不一致；不對其他 release 任意補收')
    if summary.get('target_valid_monthends_per_PE_or_PB')!=30 or summary.get('requests_executed')!=0:
        raise ValueError('第20版預查欄位不符')
    data={n:records(source_dir/n) for n in REQUIRED if n.endswith('.csv')}
    meta={'source':src,'as_of_month':summary['existing_last_month'],'existing_first_month':summary['existing_first_month'],
          'input_sha256':{n:sha((source_dir/n).read_bytes()) for n in REQUIRED},
          'summary_snapshot':{k:summary.get(k) for k in ('ordinary_roster_candidates_not_PIT_verified','stock_branch_pairs_below_30')}}
    return data,meta

def select_tasks(data, meta, official: dict, *, max_lookback_months=60):
    """Fail closed on IPO-before-listing, nonpositive, and unsupported scope."""
    asof=meta['as_of_month']; first=ymonth(month_ord(asof)-max_lookback_months+1)
    src01={ (r['market'].lower().strip(),r['symbol'].strip()):r for r in data[REQUIRED[1]] }
    below={}
    for r in data[REQUIRED[3]]:
        key=(r['market'].lower().strip(),r['symbol'].strip(),r['branch'].strip().lower())
        if key[2] not in ('pe','pb'): continue
        if int(float(r['shortfall']))>0: below[key]=r
    gaps=defaultdict(set)
    for r in data[REQUIRED[2]]:
        key=(r['market'].lower().strip(),r['symbol'].strip(),r['branch'].strip().lower())
        if key in below and r.get('gap_reason') in ('MISSING_MONTH_ROW','NULL_MULTIPLE'):
            gaps[key].add(r['month'])
    financial=defaultdict(set)
    for r in data[REQUIRED[4]]:
        market,symbol=r['market'].lower().strip(),r['symbol'].strip()
        dataset={'TaiwanStockFinancialStatements':'income','TaiwanStockBalanceSheet':'balance'}.get(r.get('dataset'))
        if dataset and any((market,symbol,branch) in below for branch in ('pe','pb')):
            financial[(market,symbol,dataset)].update(q.strip() for q in str(r.get('missing_quarters','')).split(',') if re.fullmatch(r'\d{4}Q[1-4]',q.strip()))
    grouped=defaultdict(lambda: {'branches':set(),'months':set(),'earlier':False})
    blocked=[]
    for (market,symbol,branch),r in sorted(below.items()):
        k=(market,symbol); ordinary=official.get(k)
        base=src01.get(k)
        if not ordinary or not base or str(base.get('has_any_valuation_evidence','')).lower()!='true':
            blocked.append({'market':market,'symbol':symbol,'branch':branch,'reason':'NOT_CONFIRMED_CURRENT_ORDINARY_OR_NO_EXISTING_VALUATION'});continue
        listing=ordinary.get('listing_date')
        initial_month=base.get('first_observed_month') or meta.get('existing_first_month','')
        current_gaps=[]
        for month in gaps.get((market,symbol,branch),set()):
            if month_ord(month)<month_ord(first) or month_ord(month)>month_ord(asof): continue
            # IPO unknown: only query months at/after existing documented valuation observations.
            if listing and last_day(month)<listing: continue
            if not listing and initial_month and month_ord(month)<month_ord(initial_month):continue
            current_gaps.append(month)
        state=grouped[k]
        state['branches'].add(branch)
        state['months'].update(current_gaps)
        # Earlier-history is only eligible with authoritative current listing date available;
        # not an assertion that PIT or historical ordinary-share classification is proven.
        existing_first=meta.get('existing_first_month','2023-10')
        if r.get('may_need_pre_existing_history_before') and listing and listing<first_day(existing_first):
            prior_months=[ymonth(n) for n in range(month_ord(first),month_ord(existing_first)) if last_day(ymonth(n))>=listing]
            # Need only enough candidate months to fill shortfall + capped buffer. Latest first.
            need=min(len(prior_months),int(float(r['shortfall']))+4)
            state['months'].update(prior_months[-need:]);state['earlier']=bool(need)
        if not current_gaps and not state['earlier']:
            blocked.append({'market':market,'symbol':symbol,'branch':branch,'reason':'NO_SAFE_MISSING_OR_NULL_MONTHS_OR_PRELISTING_HISTORY'})
    tasks=[]
    for (market,symbol),r in sorted(grouped.items()):
        if not r['months']:continue
        mn=min(r['months']);mx=max(r['months'])
        tasks.append({'kind':'valuation','dataset':DATASETS['valuation'],'market':market,'symbol':symbol,
                      'start_date':first_day(mn),'end_date':last_day(mx),'target_months':sorted(r['months']),
                      'branches':sorted(r['branches']),'listing_date':official[(market,symbol)].get('listing_date'),
                      'research_only':True,'reason':'MISSING_OR_NULL_MONTH_REVIEW'})
    # Only financial quarters relevant to the observed / requested valuation study interval;
    # don't infer missing data from absent published_at; don't request IPO-prelisting quarters.
    for (market,symbol,kind),qset in sorted(financial.items()):
        if (market,symbol) not in grouped or (market,symbol) not in official:continue
        base=src01[(market,symbol)]
        listing=official[(market,symbol)].get('listing_date')
        # Without IPO date, conservative earliest observed valuation month as lower bound.
        visible=base.get('first_observed_month') or meta.get('existing_first_month','')
        lower=(min(grouped[(market,symbol)]['months']) if grouped[(market,symbol)]['months'] else visible)
        start=ymonth(max(month_ord(first),month_ord(lower)-12)) # financial warmup
        valid=[]
        for q in sorted(qset):
            d=quarter_end(q)
            if d>last_day(asof): continue
            if month_ord(d[:7])<month_ord(start):continue
            if listing and d<listing:continue
            if not listing and visible and month_ord(d[:7])<month_ord(visible):continue
            valid.append(q)
        if not valid:continue
        tasks.append({'kind':kind,'dataset':DATASETS[kind],'market':market,'symbol':symbol,
                      'start_date':quarter_end(min(valid)),'end_date':quarter_end(max(valid)),
                      'target_quarters':valid,'listing_date':listing,
                      'research_only':True,'reason':'WHOLE_STATEMENT_QUARTER_ABSENT_SOURCE_CHECK'})
    return sorted(tasks,key=lambda r:(r['symbol'],r['dataset'],r['start_date'])),blocked

def task_id(task):
    x={k:task[k] for k in ('dataset','symbol','start_date','end_date')}
    return sha(json.dumps(x,sort_keys=True).encode())[:20]

def verify_receipt(root: Path, task):
    tid=task_id(task);receipt=root/'receipts'/f'{tid}.json';raw=root/'raw'/f'{tid}.json'
    if not receipt.is_file() or not raw.is_file():return None
    try:
        r=json.loads(receipt.read_text('utf-8'))
        if r['task_id']!=tid or r['dataset']!=task['dataset'] or r['symbol']!=task['symbol'] or r['raw_sha256']!=sha(raw.read_bytes()):return None
        if r.get('status') not in ('SUCCESS','EMPTY_SOURCE_RESPONSE'):return None
        j=json.loads(raw.read_bytes())
        if str(j.get('status'))!='200' or not isinstance(j.get('data'),list):return None
        return r
    except (ValueError,KeyError):return None

def response_kind(status_code, payload):
    if status_code in (401,403) or str(payload.get('status')) in ('401','403'):return 'ACCESS_DENIED'
    if status_code in (402,429) or str(payload.get('status')) in ('402','429'):return 'QUOTA_PAUSED'
    if status_code>=500:return 'RETRYABLE_HTTP'
    if status_code!=200:return 'ACCESS_OR_HTTP_ERROR'
    if not isinstance(payload,dict) or str(payload.get('status'))!='200' or not isinstance(payload.get('data'),list):return 'SCHEMA_OR_SOURCE_ERROR'
    return 'EMPTY_SOURCE_RESPONSE' if not payload['data'] else 'SUCCESS'

def safe_finmind_response(body:bytes, task):
    try:j=json.loads(body)
    except (ValueError,UnicodeDecodeError):return 'SCHEMA_OR_SOURCE_ERROR',0
    if not isinstance(j,dict):return 'SCHEMA_OR_SOURCE_ERROR',0
    if j.get('status') not in (200,'200'):
        return response_kind(200,{'status':int(j.get('status',0)) if str(j.get('status','')).isdigit() else 0}),0
    rows=j.get('data')
    if not isinstance(rows,list):return 'SCHEMA_OR_SOURCE_ERROR',0
    for item in rows:
        if not isinstance(item,dict) or str(item.get('stock_id','')).strip()!=task['symbol']:
            return 'SCHEMA_OR_SOURCE_ERROR',0
        if task['kind']=='valuation':
            date=item.get('date')
            if not date or not str(date)[:10].startswith(('19','20')):
                return 'SCHEMA_OR_SOURCE_ERROR',0
            if not ('PER' in item and 'PBR' in item):return 'SCHEMA_OR_SOURCE_ERROR',0
        else:
            if not {'date','type','value'}.issubset(item):return 'SCHEMA_OR_SOURCE_ERROR',0
    return ('SUCCESS' if rows else 'EMPTY_SOURCE_RESPONSE'),len(rows)

def persist_response(root, task, response, status, count):
    tid=task_id(task);raw=root/'raw'/f'{tid}.json';receipt=root/'receipts'/f'{tid}.json'
    atomic_bytes(raw,response)
    r={'task_id':tid,'status':status,'dataset':task['dataset'],'kind':task['kind'],
       'market':task['market'],'symbol':task['symbol'],'start_date':task['start_date'],
       'end_date':task['end_date'],'retrieved_at_utc':utc(),'raw_sha256':sha(response),
       'raw_bytes':len(response),'rows':count,'historical_pit_verified':False,
       'publication_allowed':False,'source_authorization':'REVIEW_REQUIRED',
       'month_end_official_verified':False,'request_auth':'BEARER_HEADER_NOT_PERSISTED'}
    atomic_json(receipt,r)
    return r

def run_fetch(root, tasks, *, token, max_calls_per_run=25, sleep_seconds=8.0,
              session=None, sleeper=time.sleep, retry_once=False):
    import requests
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    if not token:raise RuntimeError('FinMind Token 未設定；無法發送補收請求')
    if not (1<=max_calls_per_run<=500):raise ValueError('MAX_CALLS_PER_RUN 必須在 1..500')
    if sleep_seconds < 7.2:raise ValueError('避免超出免費額度，請將請求間隔維持 >=7.2 秒')
    ses=session or requests.Session()
    actual=0;statuses=[];quota_stopped=False;last_call=None
    for task in tasks:
        cache=verify_receipt(root,task)
        if cache:
            statuses.append({'task_id':task_id(task),'status':'CACHED_'+cache['status']});continue
        if actual>=max_calls_per_run or quota_stopped: break
        # Avoid token in query / request URL. Do not log exception with request path.
        params={'dataset':task['dataset'],'data_id':task['symbol'],
                'start_date':task['start_date'],'end_date':task['end_date']}
        headers={'Authorization':'Bearer '+token,'User-Agent':'Janus-Valuation-04-Research-Backfill/2.1'}
        if last_call is not None:sleeper(max(0,sleep_seconds-(time.monotonic()-last_call)))
        attempts=2 if retry_once else 1
        for attempt in range(attempts):
            if actual>=max_calls_per_run:break
            last_call=time.monotonic();actual+=1
            try:
                r=ses.get(BASE_URL,params=params,headers=headers,timeout=(12,60))
                payload=r.content
                try: parsed=json.loads(payload)
                except (ValueError,UnicodeDecodeError):parsed={}
                if isinstance(parsed,dict) and str(parsed.get('status')) in ('402','429'):
                    kind='QUOTA_PAUSED'
                else:
                    kind=response_kind(r.status_code, parsed if isinstance(parsed,dict) else {})
                if kind in ('SUCCESS','EMPTY_SOURCE_RESPONSE'):
                    kind,n=safe_finmind_response(payload,task)
                    if kind in ('SUCCESS','EMPTY_SOURCE_RESPONSE'):
                        persist_response(root,task,payload,kind,n)
                if kind=='QUOTA_PAUSED':quota_stopped=True
            except requests.RequestException:
                kind='RETRYABLE_NETWORK' # never print potentially token-bearing URL
            statuses.append({'task_id':task_id(task),'dataset':task['dataset'],
                             'symbol':task['symbol'],'status':kind,'attempt':attempt+1})
            atomic_json(root/'last_progress.json',{'updated_at':utc(),'http_calls_this_run':actual,
                      'last_symbol':task['symbol'],'last_dataset':task['dataset'],
                      'last_status':kind,'quota_paused':quota_stopped,
                      'publication_allowed':False})
            if kind not in ('RETRYABLE_HTTP','RETRYABLE_NETWORK') or attempt==attempts-1:break
            sleeper(max(15.0,sleep_seconds));last_call=time.monotonic()
        if quota_stopped or actual>=max_calls_per_run: break
    return {'actual_http_calls_this_run':actual,'cached_verified':sum(1 for s in statuses if s['status'].startswith('CACHED_')),
            'success_or_empty_this_run':sum(1 for s in statuses if s['status'] in ('SUCCESS','EMPTY_SOURCE_RESPONSE')),
            'quota_paused':quota_stopped,'attempts':statuses[-60:]}

def generate_reports(root, tasks, base_data, *, run_summary=None):
    receipts=[verify_receipt(Path(root),t) for t in tasks]
    statuses=[{'task_id':task_id(t),'symbol':t['symbol'],'dataset':t['dataset'],
              'kind':t['kind'],'start_date':t['start_date'],'end_date':t['end_date'],
              'status':('FINISHED_'+r['status'] if r else 'PENDING_OR_ERROR'),
              'rows':(r.get('rows',0) if r else 0)} for t,r in zip(tasks,receipts)]
    atomic_csv(root/'01_task_status.csv',statuses,['task_id','symbol','dataset','kind','start_date','end_date','status','rows'])
    # Only candidate positive PE/PB, not verified true exchange month-end or safe-to-train PIT.
    base={(r['market'].lower().strip(),r['symbol'].strip()):r for r in base_data[REQUIRED[1]]}
    more=defaultdict(lambda:{'pe':set(),'pb':set()})
    allowed={(r['market'].lower().strip(),r['symbol'].strip(),r['branch'].strip().lower(),r['month'])
      for r in base_data[REQUIRED[2]] if r.get('gap_reason') in ('MISSING_MONTH_ROW','NULL_MULTIPLE')}
    val_rows=[];financial_rows=[]
    for t,r in zip(tasks,receipts):
        if not r or r.get('status')!='SUCCESS':continue
        blob=json.loads((root/'raw'/(task_id(t)+'.json')).read_bytes())['data']
        if t['kind']=='valuation':
            by_month=defaultdict(list)
            for x in blob:
                date=str(x.get('date',''))[:10]
                try:d=dt.date.fromisoformat(date)
                except ValueError: continue
                if not (t['start_date']<=date<=t['end_date']):continue
                if t.get('listing_date') and date<t['listing_date']:continue
                by_month[date[:7]].append((date,x))
            for month,pairs in sorted(by_month.items()):
                # use last actual observation from this HTTP response, no synthetic monthends
                date,x=max(pairs,key=lambda y:y[0])
                for branch,col in (('pe','PER'),('pb','PBR')):
                    v=x.get(col)
                    try:v=float(v)
                    except (ValueError,TypeError):v=math.nan
                    valid=math.isfinite(v) and v>0
                    candidate=(t['market'],t['symbol'],branch,month) in allowed or (month in t.get('target_months',[]) and month<base[(t['market'],t['symbol'])].get('first_observed_month','9999-99'))
                    if valid and candidate:more[(t['market'],t['symbol'])][branch].add(month)
                    val_rows.append({'market':t['market'],'symbol':t['symbol'],'month':month,'observed_date':date,
                                     'branch':branch,'multiple':v if math.isfinite(v) else '',
                                     'positive':valid,'candidate_month':candidate,'source_raw_sha256':r['raw_sha256'],
                                     'status':'RESEARCH_CANDIDATE_PIT_AND_MONTHEND_UNVERIFIED'})
        else:
            per_q=defaultdict(int)
            for row in blob:
                date=str(row.get('date',''))[:10]
                if date:
                    per_q[date[:7]]+=1
            for q in t['target_quarters']:
                financial_rows.append({'market':t['market'],'symbol':t['symbol'],'dataset':t['dataset'],
                                       'fiscal_quarter':q,'source_rows_with_same_quarter_end':per_q.get(quarter_end(q)[:7],0),
                                       'status':'RAW_RESPONSE_ONLY_SCOPE_UNIT_PIT_UNVERIFIED','source_raw_sha256':r['raw_sha256']})
    atomic_csv(root/'02_valuation_recovered_month_candidates.csv',val_rows,
       ['market','symbol','month','observed_date','branch','multiple','positive','candidate_month','source_raw_sha256','status'])
    atomic_csv(root/'03_financial_source_check.csv',financial_rows,
       ['market','symbol','dataset','fiscal_quarter','source_rows_with_same_quarter_end','status','source_raw_sha256'])
    scores=[]
    for key,v in sorted(base.items()):
        for branch in ('pe','pb'):
            n=int(float(v.get(f'{branch}_positive_months') or 0));additional=len(more[key][branch]);
            scores.append({'market':key[0],'symbol':key[1],'branch':branch,'existing_positive_months':n,
                           'additional_positive_month_candidates':additional,
                           'possible_positive_months_not_pit_verified':n+additional,
                           'precheck_30_positive_months':n+additional>=30,
                           'model_ready':False,'publication_allowed':False})
    atomic_csv(root/'04_postfetch_positive_coverage_precheck.csv',scores,
       ['market','symbol','branch','existing_positive_months','additional_positive_month_candidates',
        'possible_positive_months_not_pit_verified','precheck_30_positive_months','model_ready','publication_allowed'])
    summary={'status':'FETCH_ATTEMPTED_NOT_MODEL_READY','tasks_total':len(tasks),
      'tasks_success':sum(1 for r in receipts if r and r.get('status')=='SUCCESS'),
      'tasks_empty':sum(1 for r in receipts if r and r.get('status')=='EMPTY_SOURCE_RESPONSE'),
      'tasks_pending':sum(1 for r in receipts if not r),
      'candidate_positive_months_added':sum(len(x['pe'])+len(x['pb']) for x in more.values()),
      'run_stats':run_summary or {},'publication_allowed':False,
      'limitations':['FinMind daily last observed is not certified exchange month-end','positive multiples do not certify PIT, units or financial comparability','historical source versions are not reconstructed'],
      'updated_utc':utc()}
    atomic_json(root/'00_run_summary.json',summary)
    return summary


In [ ]:
# 3. 固定第20版的六檔輸入／資料版本；先不花 FinMind 額度
inputs,source_meta=read_sources(SOURCE_DIR)
RELEASE=ROOT/'資料源'/'JanusDataset_v3_202610'/'canonical'/source_meta['source']['release']
manifest=RELEASE/'manifest.json'
assert manifest.is_file() and sha(manifest.read_bytes())==EXPECTED_MANIFEST_SHA, '原研究 release manifest 已移動或內容變動，停止'
current=ROOT/'資料源'/'JanusDataset_v3_202610'/'canonical'/'CURRENT.json'
if current.is_file() and sha(current.read_bytes())!=EXPECTED_CURRENT_SHA:
    print('注意：CURRENT 已前進；此輪仍對凍結的原 release/第20版缺口，不追隨新 CURRENT。')
print('已驗輸入數：',len(REQUIRED),'來源：',source_meta['source'])


In [ ]:
# 4. 查今天官方普通股主檔；只作研究候選，不能冒充歷史 PIT
company_roster={}; official_receipts=[]
for market,url in OFFICIAL.items():
    r=requests.get(url,timeout=(12,45),headers={'User-Agent':'Janus-Valuation-04-Research-Backfill/2.1'})
    r.raise_for_status()
    approved=parse_official_master(market,r.content) # schema 有變會停止，不濫抓
    company_roster.update(approved)
    official_receipts.append({'market':market,'url':url,'response_sha256':sha(r.content),
                              'checked_utc':utc(),'candidate_count':len(approved)})
print('當前官方公司主檔候選（非歷史普通股PIT）：',len(company_roster))
tasks,blocked=select_tasks(inputs,source_meta,company_roster,max_lookback_months=MAX_LOOKBACK_MONTHS)
if not tasks:raise RuntimeError('沒有通過來源/資格初篩的可安全查詢任務，停止；請檢查 blocked 與來源 CSV，不會假稱有抓到資料')
DEST.mkdir(parents=True,exist_ok=True)
plan_payload={'version':'valuation-finmind-min30-fetch-v2.1',
              'source_meta':source_meta,'tasks':tasks,'max_lookback_months':MAX_LOOKBACK_MONTHS,
              'publication_allowed':False,'note':'只允許真實缺月/整季無資料的有界查核'}
plan_fingerprint=sha(json.dumps(plan_payload,sort_keys=True,ensure_ascii=False).encode())
plan_file=DEST/'frozen_plan.json'
if plan_file.is_file():
    frozen=json.loads(plan_file.read_text('utf-8'))
    if frozen.get('fingerprint')!=plan_fingerprint:
        raise RuntimeError('已有不同任務快照；為防止跨版本混合，請用新 DEST 目錄重建，不要覆蓋舊補收收據')
else:
    atomic_json(plan_file,{'fingerprint':plan_fingerprint,'plan':plan_payload})
atomic_json(DEST/'official_master_readback.json',{'evidence':official_receipts,
                'listing_date_from_current_master_not_history_pit':True})
atomic_csv(DEST/'plan_tasks.csv',
  [{'task_id':task_id(t),'symbol':t['symbol'],'market':t['market'],'dataset':t['dataset'],
    'kind':t['kind'],'start_date':t['start_date'],'end_date':t['end_date'],
    'target_count':len(t.get('target_months',t.get('target_quarters',[])))} for t in tasks],
    ['task_id','symbol','market','dataset','kind','start_date','end_date','target_count'])
atomic_csv(DEST/'blocked_source_checks.csv',blocked,['market','symbol','branch','reason'])
print('計畫任務：',len(tasks),'（PER/PBR、財報合計；一檔同資料集日期合併）')
print('待人工核查／未安排 API 的分支：',len(blocked))
print('前8項：',[(x['symbol'],x['dataset'],x['start_date'],x['end_date']) for x in tasks[:8]])


In [ ]:
# 5. 真實補收：預設 TRUE；這個儲存格確實呼叫 FinMind，而不是只寫缺口清單
if not RUN_FETCH:
    raise RuntimeError('RUN_FETCH=False：你目前只在預覽。要實際補收請將第1格改為 True。')
TOKEN=getpass.getpass('請貼上 FinMind 免費 Token（僅暫存 Colab RAM，不寫入 Drive）：').strip()
assert TOKEN, '需要 FinMind Token；請登入 FinMind 取得。'
print('開始使用 FinMind API，最多',MAX_CALLS_PER_RUN,'次 GET；有快取的會跳過。')
run_stats=run_fetch(DEST,tasks,token=TOKEN,max_calls_per_run=MAX_CALLS_PER_RUN,
                   sleep_seconds=SLEEP_SECONDS,retry_once=RETRY_ONCE)
del TOKEN
print('**本輪真實 HTTP GET 次數：**',run_stats['actual_http_calls_this_run'])
print('SHA 驗證後跳過已完成任務：',run_stats['cached_verified'])
print('本輪有原始回應保存：',run_stats['success_or_empty_this_run'])
print('免費額度暫停：',run_stats['quota_paused'])


In [ ]:
# 6. 從 raw+receipt 真正讀回，對比原有正值月末與下一輪未完成任務。
post=generate_reports(DEST,tasks,inputs,run_summary=run_stats)
print(json.dumps(post,ensure_ascii=False,indent=2))
print('來源候選：',DEST/'raw')
print('結果報表：',[(p.name,p.stat().st_size) for p in DEST.glob('*.csv')])
print('完成不是模型 ready：所有回應仍需檢 PIT、合併口徑、月底判定與來源使用權利。')


## 執行完後怎麼用

- 看 `00_run_summary.json`：`tasks_success` 是真正抓到且保存來源資料的任務數，`tasks_empty` 是來源回空陣列（不是「已補滿」），`tasks_pending` 是尚未成功／未取得資料的任務。`run_stats.actual_http_calls_this_run` 是**這一輪真實 GET 次數**。
- 看 `01_task_status.csv`、`raw/` 與 `receipts/`：每一筆成功回應以 SHA256 留存。**不要把 Token 放進輸出、截圖或訊息。**
- 看 `02_valuation_recovered_month_candidates.csv` 與 `04_postfetch_positive_coverage_precheck.csv`：只是本次可觀測的正 PE/PB 月份，`model_ready` 一律 `False`。
- 再按「執行全部」：上輪成功且 SHA 完整的項目會直接讀快取，重試其他任務；**不需要重新人工列清冊**。當 FinMind 回 402／429 額度限制時會停止本輪，可待配額恢復後重新執行。
- 權限與來源保護：只有 `janusChatGPT` 根下 `模型研究/valuation/backfill_candidates/FINMIND_MIN30_REAL_20261024` 寫入研究產物，**不會改 canonical、Risk ACTIVE 或 Mart**。如本次發現執行計畫/官方公司主檔比上輪有差異而停止，請保留舊目錄、使用新版本輸出目錄，勿刪除舊收據。
